# MoE Model Compatibility Test

Testing multiple approaches to find a working MoE model for Kaggle (16GB VRAM).

In [2]:
!pip install transformers>=4.41.0 accelerate datasets peft torch>=2.2.0 tabulate -q

In [6]:
!pip install einops

  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)


In [8]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import torch
import gc

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

def load_phi_tiny_moe(model_id="microsoft/Phi-tiny-moe-instruct"):
    """Load Phi-tiny-moe without requiring flash_attn on Windows."""
    try:
        # Prefer native Transformers implementation (no remote-code dependency checks).
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16,
            device_map='auto',
            trust_remote_code=False,
            attn_implementation='eager',
        )
    except Exception as native_err:
        print(f"Native load failed: {native_err}")
        raise RuntimeError(
            "Could not load Phi-tiny-moe without remote code. "
            "Update transformers to a newer version that includes PhiMoE support, "
            "or choose another MoE model in Approach 2."
        ) from native_err

def test_model(model, tokenizer, model_name):
    """Quick test to see if model generates reasonable output."""
    print(f"\n{'='*60}")
    print(f"Testing: {model_name}")
    print(f"{'='*60}")
    
    # Check vocab match
    model_vocab = model.config.vocab_size
    tok_vocab = len(tokenizer)
    print(f"Model vocab: {model_vocab}, Tokenizer vocab: {tok_vocab}")
    print(f"Match: {'YES' if tok_vocab <= model_vocab else 'NO - MISMATCH!'}")
    
    # Test generation
    device = next(model.parameters()).device
    test_prompt = "def add(a, b):\n    return"
    
    inputs = tokenizer(test_prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        out = model.generate(
            inputs.input_ids,
            max_new_tokens=30,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    completion = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"\nPrompt: {test_prompt}")
    print(f"Output: {completion}")
    
    # Check if garbage
    new_text = completion[len(test_prompt):]
    is_garbage = (
        new_text.count(')') > 10 or
        new_text.count('\\') > 5 or
        len(set(new_text.split())) < 2 or
        '...' in new_text or
        new_text.strip() == ''
    )
    
    if is_garbage:
        print("\n❌ GARBAGE OUTPUT - Model not working")
        return False
    else:
        print("\n✅ LOOKS GOOD - Model generates reasonable output!")
        return True

cleanup()

## Approach 1: Fix Phi-tiny-moe Tokenizer

In [ ]:
# ── Approach 1A: Try different tokenizers for Phi-tiny-moe ────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "microsoft/Phi-tiny-moe-instruct"

print("Loading Phi-tiny-moe model...")
model = load_phi_tiny_moe(MODEL_ID)
print(f"Model vocab_size: {model.config.vocab_size}")

# Try different tokenizers
tokenizer_candidates = [
    "microsoft/Phi-tiny-moe-instruct",  # Original
    "microsoft/phi-2",                   # Phi-2 tokenizer
    "microsoft/phi-1_5",                 # Phi-1.5 tokenizer  
    "codellama/CodeLlama-7b-hf",         # CodeLlama (similar vocab size)
]

working_tokenizer = None
for tok_name in tokenizer_candidates:
    try:
        print(f"\nTrying tokenizer: {tok_name}")
        tokenizer = AutoTokenizer.from_pretrained(tok_name, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        print(f"  Tokenizer vocab: {len(tokenizer)}")
        
        if len(tokenizer) <= model.config.vocab_size:
            if test_model(model, tokenizer, f"Phi-tiny-moe + {tok_name}"):
                working_tokenizer = tok_name
                print(f"\n🎉 FOUND WORKING COMBINATION!")
                break
    except Exception as e:
        print(f"  Error: {e}")

if not working_tokenizer:
    print("\n❌ No working tokenizer found for Phi-tiny-moe")
    del model
    cleanup()

Loading Phi-tiny-moe model...


Encountered exception while importing flash_attn: No module named 'flash_attn'


ImportError: This modeling file requires the following packages that were not found in your environment: flash_attn. Run `pip install flash_attn`

In [ ]:
# ── Approach 1B: Add Phi-3 special tokens to tokenizer ────────────────────────
# The 53 token difference might be Phi-3 style special tokens!
# We add tokens to the TOKENIZER to match the MODEL (don't resize model!)

if 'model' not in dir() or model is None:
    print("Loading model again...")
    model = load_phi_tiny_moe("microsoft/Phi-tiny-moe-instruct")

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-tiny-moe-instruct", trust_remote_code=True)
print(f"Original tokenizer vocab: {len(tokenizer)}")
print(f"Model expects: {model.config.vocab_size}")

diff = model.config.vocab_size - len(tokenizer)
print(f"Difference: {diff} tokens")

# Add Phi-3 style special tokens (from the GitHub solution)
phi3_special_tokens = [
    "<|system|>",
    "<|user|>", 
    "<|assistant|>",
    "<|end|>",
    "<|endoftext|>",
    "<|pad|>",
]

# Also add extra tokens to fill remaining gap
remaining = diff - len(phi3_special_tokens)
if remaining > 0:
    phi3_special_tokens.extend([f"<|reserved_{i}|>" for i in range(remaining)])

print(f"\nAdding {len(phi3_special_tokens[:diff])} special tokens to tokenizer...")
num_added = tokenizer.add_special_tokens({"additional_special_tokens": phi3_special_tokens[:diff]})
print(f"Added {num_added} tokens, new tokenizer vocab size: {len(tokenizer)}")
print(f"Model vocab size (unchanged): {model.config.vocab_size}")

# Verify match
if len(tokenizer) == model.config.vocab_size:
    print("✅ Vocab sizes now match!")
else:
    print(f"⚠️ Still mismatched: tokenizer={len(tokenizer)}, model={model.config.vocab_size}")

if tokenizer.pad_token is None:
    tokenizer.pad_token = "<|pad|>" if "<|pad|>" in tokenizer.get_vocab() else tokenizer.eos_token

test_model(model, tokenizer, "Phi-tiny-moe + Phi-3 special tokens")

del model
cleanup()

In [ ]:
# ── Approach 1C: Use Phi-3 instruction format ─────────────────────────────────
# The model may only work with properly formatted prompts!

if 'model' not in dir() or model is None:
    print("Loading model again...")
    model = load_phi_tiny_moe("microsoft/Phi-tiny-moe-instruct")
    tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-tiny-moe-instruct", trust_remote_code=True)
    
    # Add missing special tokens
    diff = model.config.vocab_size - len(tokenizer)
    if diff > 0:
        tokens = [f"<|reserved_{i}|>" for i in range(diff)]
        tokenizer.add_special_tokens({"additional_special_tokens": tokens})

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Phi-3 instruction format from the GitHub solution
def format_phi3_prompt(instruction):
    """Format a prompt using Phi-3 instruction template."""
    return f"""<|system|>
You are a helpful AI assistant that writes Python code.<|end|>
<|user|>
{instruction}<|end|>
<|assistant|>
"""

# Test with formatted prompt
test_prompt = "Write a function to add two numbers"
formatted = format_phi3_prompt(test_prompt)
print(f"Testing with Phi-3 instruction format:")
print(f"Formatted prompt:\n{formatted}")

device = next(model.parameters()).device
inputs = tokenizer(formatted, return_tensors="pt").to(device)

with torch.no_grad():
    out = model.generate(
        inputs.input_ids,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

completion = tokenizer.decode(out[0], skip_special_tokens=False)
print(f"\nOutput:\n{completion}")

# Also test code completion style
print("\n" + "="*60)
print("Testing code completion style:")
code_prompt = format_phi3_prompt("Complete this Python function:\ndef add(a, b):\n    return")
inputs = tokenizer(code_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    out = model.generate(
        inputs.input_ids,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

completion = tokenizer.decode(out[0], skip_special_tokens=False)
print(f"Output:\n{completion}")

del model
cleanup()

## Approach 2: Try Other Small MoE Models

In [ ]:
# ── Approach 2A: OLMoE-1B-7B (1B active, 7B total) ────────────────────────────
# Small open MoE model - BEST BET for Kaggle!
cleanup()

# Try both base and instruct versions
for MODEL_ID in ["allenai/OLMoE-1B-7B-0924-Instruct", "allenai/OLMoE-1B-7B-0924"]:
    print(f"\n{'='*60}")
    print(f"Testing {MODEL_ID}...")
    print(f"{'='*60}")
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.bfloat16,
            device_map='auto',
            trust_remote_code=True,
        )
        
        print(f"Model loaded! VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        print(f"Num experts: {getattr(model.config, 'num_experts', 'N/A')}")
        print(f"Experts per token: {getattr(model.config, 'num_experts_per_tok', 'N/A')}")
        
        # Check MoE structure
        print("\nMoE Architecture Check:")
        for name, module in model.named_modules():
            if 'expert' in name.lower() or 'moe' in name.lower() or 'router' in name.lower():
                print(f"  {name}: {type(module).__name__}")
                break
        
        if test_model(model, tokenizer, MODEL_ID):
            print(f"\n🎉 {MODEL_ID} WORKS! Use this for the thesis.")
            # Don't delete - keep for further testing
            break
        else:
            del model
            cleanup()
        
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        if 'model' in dir() and model is not None:
            del model
        cleanup()

In [ ]:
# ── Approach 2B: DeepSeek-MoE-16b with CPU offload ────────────────────────────
cleanup()

MODEL_ID = "deepseek-ai/deepseek-moe-16b-base"
print(f"Testing {MODEL_ID} with CPU offload...")

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Use CPU offload to fit in memory
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        trust_remote_code=True,
        offload_folder="offload",
    )
    
    print(f"Model loaded! VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    print(f"Num experts: {getattr(model.config, 'num_experts', 'N/A')}")
    
    test_model(model, tokenizer, MODEL_ID)
    
except Exception as e:
    print(f"Error: {e}")
finally:
    if 'model' in dir():
        del model
    cleanup()

In [ ]:
# ── Approach 2C: Switch-Transformers (Google's MoE) ───────────────────────────
cleanup()

# Switch-C is encoder-decoder, let's try a smaller variant
MODEL_ID = "google/switch-base-8"  # 8 experts, smaller model
print(f"Testing {MODEL_ID}...")

try:
    from transformers import SwitchTransformersForConditionalGeneration, T5Tokenizer
    
    tokenizer = T5Tokenizer.from_pretrained(MODEL_ID)
    model = SwitchTransformersForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map='auto',
    )
    
    print(f"Model loaded! VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    print(f"This is encoder-decoder (T5-style), not decoder-only")
    print(f"May not be ideal for code completion task.")
    
except Exception as e:
    print(f"Error: {e}")
finally:
    if 'model' in dir():
        del model
    cleanup()

In [ ]:
# ── Approach 2D: Qwen MoE with aggressive memory settings ─────────────────────
cleanup()

MODEL_ID = "Qwen/Qwen1.5-MoE-A2.7B-Chat"
print(f"Testing {MODEL_ID} with memory optimizations...")

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Aggressive memory settings
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,  # float16 instead of bfloat16
        device_map='auto',
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        offload_folder="offload",
        max_memory={0: "14GB", "cpu": "30GB"},  # Reserve some VRAM
    )
    
    print(f"Model loaded! VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    print(f"Num experts: {getattr(model.config, 'num_experts', 'N/A')}")
    
    test_model(model, tokenizer, MODEL_ID)
    
except Exception as e:
    print(f"Error: {e}")
finally:
    if 'model' in dir():
        del model
    cleanup()

In [ ]:
# ── Approach 2E: JetMoE - Specifically designed to be small ───────────────────
cleanup()

MODEL_ID = "jetmoe/jetmoe-8b"  # 8B total, 2B active
print(f"Testing {MODEL_ID}...")

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        trust_remote_code=True,
    )
    
    print(f"Model loaded! VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    print(f"Num experts: {getattr(model.config, 'num_experts', 'N/A')}")
    
    test_model(model, tokenizer, MODEL_ID)
    
except Exception as e:
    print(f"Error: {e}")
finally:
    if 'model' in dir():
        del model
    cleanup()

## Summary

Run all cells above and note which models:
1. Load successfully (fit in VRAM)
2. Generate reasonable output (not garbage)
3. Have accessible MoE architecture (for applying hierarchical LoRA)

The best candidate will be used for the thesis experiments.